# 10 — Module A : Indicateur de contexte marché Euronext

**Module A** : Dashboard de 12 signaux fondamentaux scorés (-1 → +1) agrégés en orientation marché.

Ce notebook implémente le prototype du Module A décrit dans `REFLEXION_AMELIORATION_INDICATEUR.md`.

## Architecture

```
12 signaux fondamentaux → score agrégé → orientation (HAUSSIER / NEUTRE / BAISSIER)
     ↑
  4 blocs :
  1. Offre mondiale    (bilan_mondial, bilan_stocks_eu, crop_condition_eu)
  2. Offre compétiteurs (brazil_competition, brazil_supply_pressure, ukraine_supply)
  3. Demande mondiale  (china_demand, wasde_surprise, export_pace_eu)
  4. Positionnement    (cot_positioning, futures_structure, cbot_ema_basis, eur_usd)
```

**Pré-requis** : `01_ema_data_collection.ipynb` complété.

In [ ]:
import sys
from pathlib import Path

project_root = Path("../../..").resolve()
sys.path.insert(0, str(project_root / "src"))

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.3f}".format)

## 1. Chargement des features disponibles

In [ ]:
from mais.paths import FEATURES_PARQUET, TARGETS_PARQUET, RAW_DIR
from mais.utils import read_parquet

features = read_parquet(FEATURES_PARQUET)
targets = read_parquet(TARGETS_PARQUET)

ema_path = RAW_DIR / "euronext_ema" / "euronext_ema.csv"
eurusd_path = RAW_DIR / "eu_cross_assets" / "eu_cross_assets.csv"

ema_prices = pd.read_csv(ema_path, parse_dates=["Date"]).set_index("Date") if ema_path.exists() else None
eurusd = pd.read_csv(eurusd_path, parse_dates=["Date"]).set_index("Date") if eurusd_path.exists() else None

print(f"Features: {features.shape}")
print(f"EMA disponible: {ema_prices is not None}")
print(f"EUR/USD disponible: {eurusd is not None}")

## 2. Fonctions de scoring des signaux

Chaque signal est scoré entre -1 (baissier) et +1 (haussier).

In [ ]:
def score_from_zscore(z: float, cap: float = 2.0) -> float:
    """Convertit un z-score en score [-1, +1] via tanh."""
    if np.isnan(z):
        return 0.0
    return float(np.tanh(z / cap))


def score_from_stocks_use_ratio(ratio: float, mean_5y: float = 0.25, std_5y: float = 0.05) -> float:
    """Stocks/usage faible = tendu = haussier (+1)."""
    z = -(ratio - mean_5y) / max(std_5y, 0.01)  # inversé: faible ratio = bullish
    return score_from_zscore(z)


def score_from_cot_percentile(percentile: float) -> float:
    """COT CONTRARIAN: fonds très longs (percentile > 80) = bearish risk."""
    if np.isnan(percentile):
        return 0.0
    # Contrarian: si fonds très longs → score négatif (risque retournement)
    z = -(percentile - 50) / 25  # percentile 80 → z = -1.2 → score ≈ -0.83
    return score_from_zscore(z)


def score_from_backwardation(backwardation_flag: int, contango_flag: int) -> float:
    """Backwardation = tension physique = haussier."""
    if backwardation_flag == 1:
        return 0.8
    elif contango_flag == 1:
        return -0.5
    return 0.0


def score_from_eurusd_zscore(z: float) -> float:
    """EUR faible (z < 0) = exports EU compétitifs = soutient EMA (+)."""
    # z négatif (EUR faible) = exports EU compétitifs = haussier pour EMA
    return score_from_zscore(-z)  # inversé


def score_from_wasde_surprise(stocks_surprise: float) -> float:
    """Révision à la baisse des stocks = haussier."""
    # surprise_mt négatif = stocks révisés à la baisse = bullish
    return score_from_zscore(-stocks_surprise, cap=2.0)


print("Fonctions de scoring définies.")

# Test rapide
print(f"\nTest scores:")
print(f"  stocks/usage 0.20 vs moyenne 0.25: {score_from_stocks_use_ratio(0.20):.3f} (attendu: positif)")
print(f"  COT percentile 85: {score_from_cot_percentile(85):.3f} (attendu: négatif — contrarian)")
print(f"  Backwardation: {score_from_backwardation(1, 0):.3f} (attendu: positif)")
print(f"  EUR/USD z=-1.5 (EUR faible): {score_from_eurusd_zscore(-1.5):.3f} (attendu: positif)")

## 3. Construction du tableau de bord

In [ ]:
def compute_context_score(row: pd.Series, feature_cols: list) -> dict:
    """Calculer le score contexte pour une ligne de features."""
    signals = {}

    # ── BLOC 1 : Offre mondiale ──────────────────────────────────────────
    # Bilan mondial (WASDE world stocks/use)
    if "world_stocks_use" in row.index:
        signals["bilan_mondial"] = score_from_stocks_use_ratio(row["world_stocks_use"])
    elif "wasde_world_ending_stocks" in row.index:
        signals["bilan_mondial"] = score_from_zscore(-row.get("wasde_world_ending_stocks", 0))
    else:
        # Proxy: stocks surprise WASDE si disponible
        wasde_cols = [c for c in row.index if "wasde" in c.lower() and "surprise" in c.lower()]
        signals["bilan_mondial"] = score_from_wasde_surprise(row[wasde_cols[0]]) if wasde_cols else 0.0

    # Crop condition US (proxy EU non disponible encore)
    crop_cols = [c for c in row.index if "crop" in c.lower() and ("ge_pct" in c.lower() or "good_exc" in c.lower())]
    if crop_cols:
        # Condition dégradée = offre réduite = haussier
        ge_pct = row[crop_cols[0]]
        signals["crop_condition"] = score_from_zscore(-(ge_pct - 0.65) / 0.10)
    else:
        signals["crop_condition"] = 0.0

    # ── BLOC 2 : Positionnement marché ──────────────────────────────────
    # COT managed money
    cot_cols = [c for c in row.index if "cot" in c.lower() and "percentile" in c.lower()]
    if cot_cols:
        signals["cot_positioning"] = score_from_cot_percentile(row[cot_cols[0]] * 100)
    else:
        signals["cot_positioning"] = 0.0

    # EUR/USD
    if eurusd is not None:
        eurusd_today = eurusd["eurusd_rate"].get(row.name, np.nan)
        if not np.isnan(eurusd_today):
            hist = eurusd["eurusd_rate"].loc[:row.name]
            z_eur = (eurusd_today - hist.mean()) / max(hist.std(), 0.001)
            signals["eurusd_competitiveness"] = score_from_eurusd_zscore(z_eur)
        else:
            signals["eurusd_competitiveness"] = 0.0
    else:
        signals["eurusd_competitiveness"] = 0.0

    # Score global (poids égaux pour prototype)
    valid_scores = [v for v in signals.values() if not np.isnan(v)]
    context_score = np.mean(valid_scores) if valid_scores else 0.0

    if context_score > 0.30:
        orientation = "HAUSSIER"
    elif context_score < -0.30:
        orientation = "BAISSIER"
    else:
        orientation = "NEUTRE"

    dominant = max(signals, key=lambda k: abs(signals[k])) if signals else "N/A"

    return {
        "signals": signals,
        "context_score": context_score,
        "orientation": orientation,
        "dominant_signal": dominant,
    }


print("Fonction compute_context_score définie.")

In [ ]:
# Calculer le score pour la dernière date disponible (dashboard actuel)
last_date = features.index.max()
last_row = features.loc[last_date]

result = compute_context_score(last_row, list(features.columns))
ema_close = ema_prices["ema_close"].iloc[-1] if ema_prices is not None else None

print("=" * 60)
print(f"MAÏS EURONEXT — {last_date.date()}")
if ema_close:
    print(f"Prix EMA actuel : {ema_close:.2f} EUR/t")
print(f"\nCONTEXTE MARCHÉ : {result['orientation']} (score: {result['context_score']:+.3f})")
print(f"Driver dominant : {result['dominant_signal']}")
print()

print("SIGNAUX:")
for sig_name, sig_val in sorted(result["signals"].items(), key=lambda x: -abs(x[1])):
    arrow = "↑" if sig_val > 0.2 else ("↓" if sig_val < -0.2 else "→")
    orientation = "HAUSSIER" if sig_val > 0.2 else ("BAISSIER" if sig_val < -0.2 else "NEUTRE")
    bar = "█" * int(abs(sig_val) * 10)
    print(f"  {arrow} {orientation:<10} {sig_name:<30} [{bar:<10}] {sig_val:+.3f}")

## 4. Backtest historique du score contexte vs réalité EMA

In [ ]:
# Calculer le score sur tout l'historique (sampling hebdomadaire pour la lisibilité)
# ⚠ Ce backtest ne sert qu'à vérifier la cohérence visuelle — pas à calibrer les poids

weekly_dates = features.resample("W-MON").last().index
scores_history = []

for dt in weekly_dates[-100:]:  # 100 dernières semaines
    if dt in features.index:
        r = compute_context_score(features.loc[dt], list(features.columns))
        scores_history.append({
            "date": dt,
            "context_score": r["context_score"],
            "orientation": r["orientation"],
        })

df_hist = pd.DataFrame(scores_history).set_index("date")
print(f"Historique calculé: {len(df_hist)} semaines")

if ema_prices is not None:
    # Aligner avec les retours EMA réalisés à 40 jours
    ema_close = ema_prices["ema_close"].reindex(df_hist.index, method="nearest")
    ema_fwd40 = ema_prices["ema_close"].shift(-40).reindex(df_hist.index, method="nearest")
    df_hist["ema_return_40d"] = (ema_fwd40 / ema_close - 1).values
    df_hist["correct"] = (
        ((df_hist["context_score"] > 0) & (df_hist["ema_return_40d"] > 0)) |
        ((df_hist["context_score"] < 0) & (df_hist["ema_return_40d"] < 0))
    ).astype(float)

    da_overall = df_hist["correct"].mean()
    print(f"\nDA contexte score (directionnel, h40): {da_overall:.3f}")
    print("Note: ce chiffre N'EST PAS un vrai OOF — les poids ne sont pas calibrés")
else:
    print("\n⚠ EMA non disponible pour valider le backtest")

print("\nDistribution orientations:")
print(df_hist["orientation"].value_counts())

In [ ]:
# Visualisation dashboard historique
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# Score contexte
ax = axes[0]
colors = ["tomato" if o == "BAISSIER" else ("steelblue" if o == "HAUSSIER" else "lightgray")
          for o in df_hist["orientation"]]
ax.bar(df_hist.index, df_hist["context_score"], color=colors, alpha=0.7, width=5)
ax.axhline(0.30, color="steelblue", linestyle="--", linewidth=0.8, alpha=0.5)
ax.axhline(-0.30, color="tomato", linestyle="--", linewidth=0.8, alpha=0.5)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Score contexte")
ax.set_ylim(-1, 1)
ax.set_title("Module A — Score contexte marché hebdomadaire")

# Prix EMA si disponible
ax = axes[1]
if ema_prices is not None:
    ema_weekly = ema_prices["ema_close"].reindex(df_hist.index, method="nearest")
    ax.plot(ema_weekly.index, ema_weekly.values, color="tomato", linewidth=1.2, label="EMA (EUR/t)")
    ax.set_ylabel("EUR/tonne")
    ax.legend()
else:
    ax.text(0.5, 0.5, "Prix EMA non disponible", ha="center", va="center",
            transform=ax.transAxes, fontsize=14)
ax.set_title("Prix Euronext EMA")

plt.suptitle("Module A — Dashboard Contexte Marché Euronext", fontsize=14, fontweight="bold")
plt.tight_layout()

out = project_root / "artefacts" / "module_a_dashboard.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Sauvegardé: {out}")

## 5. Prochaines étapes Module A

Pour upgrader ce prototype en Module A complet :

1. **Ajouter les signaux EU manquants** :
   - `bilan_stocks_eu` — données EC MARS ou FranceAgriMer
   - `crop_condition_eu` — Agreste france_ge_pct
   - `brazil_competition` — données CONAB/FOB
   - `china_demand` — DCE Dalian + import parity
   - `ukraine_supply` — MinAgro corridor status

2. **Calibrer les poids** en OOF walk-forward (2010–2020) :
   - Optimiser les poids pour maximiser DA hebdomadaire h40
   - Contrainte : poids positifs, somme = 1

3. **Valider la cohérence** :
   - Quand score > 0.30 → retour h40 positif dans ≥ 60% des cas ?
   - Stability semaine → semaine

4. **Intégrer au rapport hebdomadaire** via `ops/weekly_report.py`